# Multimodal Fake News Detection Pipeline (Kaggle v2)
Optimised for Kaggle 2× T4 GPUs. Trains on **GossipCop** (text) and **Weibo** (text + image).

**v2 additions over v1:**
- Per-dataset detailed evaluation (GossipCop vs. Weibo separately)
- Weibo evaluated in **3 modality groups**: text-only, image-only ablation, text+image
- 8 additional metrics: MCC, Cohen Kappa, macro/weighted F1, per-class precision/recall
- ROC curve, confusion matrix, and training-curve plots for all groups
- Standalone `predict()` function for any input combination
- NaN-loss guard and gradient-norm monitor in training loop
- Label smoothing (0.05) to improve generalisation
- `load_and_evaluate()` to reload a saved model without re-training

### 1. Install Dependencies

In [1]:
!pip install -q dgl shap lime
!python -m spacy download en_core_web_lg -q
!python -m spacy download zh_core_web_lg -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 76.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.0/603.0 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 57.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('zh_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 2. Configure Paths
Update `DATASET_MNT` to your Kaggle dataset name.

In [2]:
import os

# UPDATE to your Kaggle dataset name
DATASET_MNT    = '/kaggle/input/datasets/krishnamoorthig/fake-news-datasets-c1'
GOSSIPCOP_PATH = os.path.join(DATASET_MNT, 'datasets', 'gossipcop_final.csv')
WEIBO_PATH     = os.path.join(DATASET_MNT, 'datasets', 'weibo_final.csv')
IMAGE_DIR      = os.path.join(DATASET_MNT, 'images')
OUTPUT_DIR     = '/kaggle/working/outputs'
KG_CACHE_PATH  = os.path.join(OUTPUT_DIR, 'kg_cache.json')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Datasets : {DATASET_MNT}')
print(f'Outputs  : {OUTPUT_DIR}')


Datasets : /kaggle/input/datasets/krishnamoorthig/fake-news-datasets-c1
Outputs  : /kaggle/working/outputs


### 3. Imports & DGL Stub

In [3]:
import sys, json, time, requests, random, types, logging, warnings, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import spacy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models
from transformers import AutoTokenizer, BertModel
import transformers
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import shap
import lime, lime.lime_text

warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
for _n in ['httpx','transformers','huggingface_hub','urllib3','filelock','matplotlib','PIL']:
    logging.getLogger(_n).setLevel(logging.ERROR)

# DGL graphbolt binary stub
if 'dgl.graphbolt' not in sys.modules:
    _fake_gb = types.ModuleType('dgl.graphbolt')
    _fake_gb.__path__ = []
    sys.modules['dgl.graphbolt'] = _fake_gb
import dgl
import dgl.nn.pytorch as dglnn


DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


### 4. Utilities (seed, metrics, logging)

In [4]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True  # safe: fixed 224x224 input


def compute_metrics(y_true, y_pred, y_prob):
    """11 metrics covering binary, macro, AUC, MCC, Kappa."""
    return {
        'accuracy':         accuracy_score(y_true, y_pred),
        'precision_binary': precision_score(y_true, y_pred, zero_division=0),
        'recall_binary':    recall_score(y_true, y_pred, zero_division=0),
        'f1_binary':        f1_score(y_true, y_pred, zero_division=0),
        'precision_macro':  precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro':     recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro':         f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted':      f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'auc_roc':          roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan'),
        'mcc':              matthews_corrcoef(y_true, y_pred),
        'cohen_kappa':      cohen_kappa_score(y_true, y_pred),
    }


def save_metrics(d, path):
    with open(path, 'w') as f: json.dump(d, f, indent=2)


def setup_logging():
    logger = logging.getLogger(); logger.setLevel(logging.DEBUG); logger.handlers.clear()
    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh = logging.FileHandler(os.path.join(OUTPUT_DIR, 'training.log'), mode='w')
    fh.setLevel(logging.DEBUG); fh.setFormatter(fmt); logger.addHandler(fh)
    ch = logging.StreamHandler(); ch.setLevel(logging.WARNING); ch.setFormatter(fmt); logger.addHandler(ch)
    return logger


### 5. Knowledge Graph Linker (ConceptNet + Wikidata + GAT)

In [5]:
if 'dgl.graphbolt' not in sys.modules:
    sys.modules['dgl.graphbolt'] = _fake_gb


class KGLinker:
    def __init__(self, lang='en', cache_path=KG_CACHE_PATH):
        self.lang = lang; self.cache_path = cache_path
        self.cache = json.load(open(cache_path,'r',encoding='utf-8')) if os.path.exists(cache_path) else {}
        self.nlp = spacy.load('en_core_web_lg' if lang == 'en' else 'zh_core_web_lg')
        self.kg_dim = 300

    def _save(self):
        os.makedirs(os.path.dirname(self.cache_path), exist_ok=True)
        with open(self.cache_path,'w',encoding='utf-8') as f: json.dump(self.cache,f,indent=2,ensure_ascii=False)

    def query_conceptnet(self, entity):
        url = f'http://api.conceptnet.io/c/{self.lang}/{entity.replace(" ","_").lower()}'
        if url in self.cache: return self.cache[url]
        try:
            r = requests.get(url, timeout=10)
            d = r.json() if r.status_code==200 else {}
            self.cache[url] = d; self._save(); time.sleep(0.5)
            return d.get('edges',[]) if isinstance(d,dict) else []
        except Exception:
            self.cache[url] = []; self._save(); time.sleep(0.5); return []

    def query_wikidata(self, entity):
        k = f'wikidata_{entity}'
        if k in self.cache: return self.cache[k]
        try:
            q = f'SELECT ?item ?itemLabel WHERE {{ ?item rdfs:label "{entity}"@{self.lang} . SERVICE wikibase:label {{ bd:serviceParam wikibase:language "{self.lang},en" }} }} LIMIT 5'
            r = requests.get('https://query.wikidata.org/sparql',params={'query':q,'format':'json'},timeout=10,headers={'User-Agent':'MisinfoDetector/1.0'})
            b = r.json().get('results',{}).get('bindings',[]) if r.status_code==200 else []
            self.cache[k] = b; self._save(); return b
        except Exception:
            self.cache[k] = []; self._save(); return []

    def build_graph(self, ents):
        if not ents: return None
        torch.manual_seed(2024)
        ns, ep = set(), []
        for e in ents:
            ns.add(e)
            for edge in (self.query_conceptnet(e) or [])[:10]:
                if isinstance(edge,dict):
                    s=edge.get('start',{}); d=edge.get('end',{})
                    sl=s.get('label','') if isinstance(s,dict) else ''; el=d.get('label','') if isinstance(d,dict) else ''
                    if sl: ns.add(sl)
                    if el: ns.add(el)
                    if sl and el: ep.append((sl,el))
            self.query_wikidata(e)
            if len(ns)>=50: break
        nodes=list(ns)[:50]
        if len(nodes)<2: return None
        n2i={n:i for i,n in enumerate(nodes)}
        pairs=[(n2i[s],n2i[d]) for s,d in ep if s in n2i and d in n2i]
        src,dst=(list(zip(*pairs)) if pairs else ([0],[0]))
        g=dgl.graph((list(src),list(dst)),num_nodes=len(nodes))
        g.ndata['feat']=torch.randn(g.num_nodes(),300)
        if g.number_of_edges()==0: g=dgl.add_self_loop(g)
        return g

    def run_gat(self, g):
        l1=dglnn.GATConv(300,300,1,feat_drop=0.6,attn_drop=0.6,allow_zero_in_degree=True)
        l2=dglnn.GATConv(300,300,1,feat_drop=0.6,attn_drop=0.6,allow_zero_in_degree=True)
        h=g.ndata['feat']; h=F.elu(l1(g,h).squeeze(1)); h=l2(g,h).squeeze(1)
        return h.mean(dim=0)

    def get_embedding(self, text):
        try:
            doc=self.nlp(text); ents=[e.text for e in doc.ents]
            if not ents: return torch.zeros(300)
            g=self.build_graph(ents)
            return self.run_gat(g).detach() if g else torch.zeros(300)
        except Exception: return torch.zeros(300)


### 6. Dataset Pre-processing and DataLoaders
`build_unified_splits()` returns `gc_test_df` and `wb_test_df` separately for per-dataset evaluation.

In [6]:
image_transform = transforms.Compose([
    transforms.Resize((224,224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
SEP = ' [SEP] '


def prepare_datasets():
    gc = pd.read_csv(GOSSIPCOP_PATH)
    gc['label']       = gc['label'].map({'real':0,'fake':1}).astype(int)
    gc['input_text']  = (gc['title'].astype(str)+SEP+gc['description'].astype(str)+SEP+gc['text'].astype(str)).str[:512]
    gc['has_image']   = False; gc['image_path'] = ''; gc['lang'] = 'en'; gc['kg_embedding'] = None
    gc_tv,gc_test     = train_test_split(gc, test_size=0.10, stratify=gc['label'], random_state=42)
    gc_train,gc_val   = train_test_split(gc_tv, test_size=0.1111, stratify=gc_tv['label'], random_state=42)

    wb = pd.read_csv(WEIBO_PATH).drop_duplicates(subset='tweet_id',keep='first')
    wb['tweet_content'] = wb['tweet_content'].fillna('')
    wb['input_text']    = wb['tweet_content'].astype(str).str[:512]
    wb['image_path']    = wb.apply(lambda r: os.path.join(IMAGE_DIR,str(r['image_files'])),axis=1)
    wb['has_image']     = wb.apply(lambda r: pd.notna(r['image_files']) and str(r['image_files']).strip()!='' and os.path.isfile(r['image_path']),axis=1)
    wb['lang']='zh'; wb['kg_embedding']=None
    wb_tv,wb_test       = train_test_split(wb, test_size=0.15, stratify=wb['label'], random_state=42)
    wb_train,wb_val     = train_test_split(wb_tv, test_size=0.1765, stratify=wb_tv['label'], random_state=42)
    return gc_train,gc_val,gc_test,wb_train,wb_val,wb_test


def build_unified_splits():
    gc_train,gc_val,gc_test,wb_train,wb_val,wb_test = prepare_datasets()
    train_df = pd.concat([gc_train,wb_train],ignore_index=True).reset_index(drop=True)
    val_df   = pd.concat([gc_val,  wb_val],  ignore_index=True).reset_index(drop=True)
    test_df  = pd.concat([gc_test, wb_test], ignore_index=True).reset_index(drop=True)
    return train_df, val_df, test_df, gc_test.reset_index(drop=True), wb_test.reset_index(drop=True)


class UnifiedFakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, df, precompute_kg=False):
        self.df = df.reset_index(drop=True)
        self.en_tok = AutoTokenizer.from_pretrained('bert-base-uncased')
        self.zh_tok = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.transform = image_transform
        if precompute_kg:
            en_l,zh_l = KGLinker(lang='en'), KGLinker(lang='zh')
            self.kg_embeddings = [
                en_l.get_embedding(self.df.iloc[i]['input_text']) if self.df.iloc[i]['lang']=='en'
                else zh_l.get_embedding(self.df.iloc[i]['input_text'])
                for i in range(len(self.df))
            ]
        else:
            self.kg_embeddings = None

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tok = self.en_tok if row['lang']=='en' else self.zh_tok
        enc = tok(row['input_text'], max_length=128, padding='max_length', truncation=True, return_tensors='pt')
        ids=enc['input_ids'].squeeze(0); mask=enc['attention_mask'].squeeze(0)
        has_img=0.; img_t=torch.zeros(3,224,224)
        if row['has_image'] and isinstance(row['image_path'],str) and row['image_path']:
            try: img_t=self.transform(Image.open(row['image_path']).convert('RGB')); has_img=1.
            except Exception: pass
        kg = self.kg_embeddings[idx] if self.kg_embeddings else torch.zeros(300)
        return dict(input_ids=ids, attention_mask=mask, image=img_t,
                    has_image=torch.tensor(has_img,dtype=torch.float32),
                    kg_embedding=kg, label=torch.tensor(int(row['label']),dtype=torch.long))


def collate_fn(batch):
    return {k: torch.stack([s[k] for s in batch]) for k in batch[0]}


### 7. Multimodal Model Architecture (mBERT + ResNet-50 + Cross-Attention + GAT)

In [7]:
class UnifiedMultimodalFakeNewsDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_zh = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.bert_zh.gradient_checkpointing_enable()
        self.resnet  = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        self.resnet.fc = nn.Identity()
        for n,p in self.resnet.named_parameters():
            if not n.startswith('layer4'): p.requires_grad = False
        self.text_proj   = nn.Linear(768, 512)
        self.image_proj  = nn.Linear(2048,512)
        self.kg_proj     = nn.Linear(300, 512)
        self.cross_attn  = nn.MultiheadAttention(512, num_heads=8, batch_first=True, dropout=0.1)
        self.gate_linear = nn.Linear(1024,512)
        self.classifier  = nn.Sequential(nn.Linear(512,128),nn.ReLU(),nn.Dropout(0.3),nn.LayerNorm(128),nn.Linear(128,2))

    def forward(self, input_ids, attention_mask, image_tensor, has_image_flag, kg_embedding, lang_mask=None):
        tf  = self.bert_zh(input_ids=input_ids,attention_mask=attention_mask).pooler_output
        imf = self.resnet(image_tensor) * has_image_flag.unsqueeze(1)
        tp  = self.text_proj(tf); ip=self.image_proj(imf); kp=self.kg_proj(kg_embedding)
        att,_=self.cross_attn(tp.unsqueeze(1),ip.unsqueeze(1),ip.unsqueeze(1))
        att=att.squeeze(1)
        gate=torch.sigmoid(self.gate_linear(torch.cat([tp,att],dim=1)))
        fused=gate*tp+(1.0-gate)*ip+kp
        return self.classifier(fused)

    def get_parameter_groups(self):
        return [
            {'params':list(self.bert_zh.parameters()),     'lr':2e-5},
            {'params':list(self.resnet.layer4.parameters()),'lr':1e-4},
            {'params':list(self.text_proj.parameters()),   'lr':5e-5},
            {'params':list(self.image_proj.parameters()),  'lr':5e-5},
            {'params':list(self.kg_proj.parameters()),     'lr':5e-5},
            {'params':list(self.cross_attn.parameters()),  'lr':5e-5},
            {'params':list(self.gate_linear.parameters()), 'lr':5e-5},
            {'params':list(self.classifier.parameters()),  'lr':5e-5},
        ]


### 8. Adversarial Attacks (FGSM + PGD)

In [8]:
def fgsm_attack(image_tensor, labels, model, input_ids, attention_mask, has_image_flag, kg_embedding, epsilon=0.008):
    was=model.training; model.eval()
    p=image_tensor.clone().detach().requires_grad_(True)
    with torch.cuda.amp.autocast(enabled=image_tensor.device.type=='cuda'):
        loss=nn.CrossEntropyLoss()(model(input_ids,attention_mask,p,has_image_flag,kg_embedding),labels)
    loss.backward()
    adv=torch.clamp(p.data+epsilon*p.grad.data.sign(),0.,1.).detach()
    if was: model.train()
    return adv


def pgd_attack(image_tensor, labels, model, input_ids, attention_mask, has_image_flag, kg_embedding, epsilon=0.016, num_iter=7):
    was=model.training; model.eval()
    step=epsilon/num_iter
    delta=torch.clamp(image_tensor+torch.zeros_like(image_tensor).uniform_(-epsilon,epsilon),0.,1.)-image_tensor
    for _ in range(num_iter):
        delta=delta.detach().requires_grad_(True)
        with torch.cuda.amp.autocast(enabled=image_tensor.device.type=='cuda'):
            loss=nn.CrossEntropyLoss()(model(input_ids,attention_mask,image_tensor+delta,has_image_flag,kg_embedding),labels)
        loss.backward()
        delta=torch.clamp(delta.data+step*delta.grad.data.sign(),-epsilon,epsilon)
        delta=torch.clamp(image_tensor+delta,0.,1.)-image_tensor
    if was: model.train()
    return torch.clamp(image_tensor+delta.detach(),0.,1.)


### 9. Training Loop (NaN guard + grad norm tracking + label smoothing)

In [9]:
def train_epoch(model, loader, optimizer, scheduler, scaler, criterion, device, accum=4):
    model.train(); rloss=0.; steps=0; gnorms=[]
    pbar=tqdm(enumerate(loader),total=len(loader),desc='Training')
    for bi,batch in pbar:
        ids=batch['input_ids'].to(device); mask=batch['attention_mask'].to(device)
        img=batch['image'].to(device);     hi=batch['has_image'].to(device)
        kg=batch['kg_embedding'].to(device);lbl=batch['label'].to(device)
        with torch.cuda.amp.autocast(enabled=device.type=='cuda'):
            cl=criterion(model(ids,mask,img,hi,kg),lbl)
        adv=fgsm_attack(img,lbl,model,ids,mask,hi,kg,epsilon=0.008)
        with torch.cuda.amp.autocast(enabled=device.type=='cuda'):
            al=criterion(model(ids,mask,adv,hi,kg),lbl)
        loss=(0.5*cl+0.5*al)/accum
        if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(set_to_none=True); continue
        scaler.scale(loss).backward()
        if (bi+1)%accum==0 or bi==len(loader)-1:
            scaler.unscale_(optimizer)
            gn=torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
            gnorms.append(gn.item()); scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        rloss+=loss.item()*accum; steps+=1
        pbar.set_postfix(avg_loss=rloss/max(steps,1), grad=f'{gnorms[-1]:.3f}' if gnorms else 'n/a')
        if device.type=='cuda' and bi%50==0: torch.cuda.empty_cache()
    return rloss/max(steps,1), float(np.mean(gnorms)) if gnorms else 0.


def validate(model, loader, criterion, device):
    model.eval(); vl=correct=total=0
    with torch.no_grad():
        for batch in loader:
            ids=batch['input_ids'].to(device); mask=batch['attention_mask'].to(device)
            img=batch['image'].to(device);     hi=batch['has_image'].to(device)
            kg=batch['kg_embedding'].to(device);lbl=batch['label'].to(device)
            logits=model(ids,mask,img,hi,kg)
            vl+=nn.CrossEntropyLoss()(logits,lbl).item()
            correct+=(logits.argmax(dim=1)==lbl).sum().item(); total+=lbl.size(0)
    return vl/max(len(loader),1), correct/max(total,1)


def train_model(model, train_loader, val_loader, device, max_epochs=30, patience=5):
    logger=logging.getLogger()
    base=model.module if hasattr(model,'module') else model
    pgs=base.get_parameter_groups()
    for g in pgs: g['weight_decay']=0.01
    opt=torch.optim.AdamW(pgs)
    total_steps=(len(train_loader)//4+1)*max_epochs
    sched=transformers.get_linear_schedule_with_warmup(opt,num_warmup_steps=1000,num_training_steps=total_steps)
    scaler=torch.cuda.amp.GradScaler(enabled=device.type=='cuda')
    criterion=nn.CrossEntropyLoss(label_smoothing=0.05)
    best=0.; noim=0; log=[]
    for epoch in range(1,max_epochs+1):
        tl,gn=train_epoch(model,train_loader,opt,sched,scaler,criterion,device)
        vl,va=validate(model,val_loader,criterion,device)
        msg=f'Epoch {epoch:02d} | train={tl:.4f} | val_loss={vl:.4f} | val_acc={va:.4f} | grad={gn:.3f}'
        print(msg); logger.info(msg)
        log.append(dict(epoch=epoch,train_loss=tl,val_loss=vl,val_acc=va,mean_grad_norm=gn))
        if va>best:
            best=va; noim=0
            state=model.module.state_dict() if hasattr(model,'module') else model.state_dict()
            torch.save(state,os.path.join(OUTPUT_DIR,'best_model.pt'))
        else:
            noim+=1
            if noim>=patience: print(f'Early stopping at epoch {epoch}'); break
    with open(os.path.join(OUTPUT_DIR,'training_log.json'),'w') as f: json.dump(log,f,indent=2)
    return best, log


### 10. Evaluation Functions
- `evaluate_loader()` — metrics on any DataLoader
- `evaluate_by_dataset()` — 6 groups: GossipCop, Weibo text-only, Weibo image-only, Weibo both, Weibo full, Combined
- `evaluate_adversarial()` — FGSM & PGD robustness

In [10]:
def _run_inference(model, loader, device):
    model.eval(); labels=[]; preds=[]; probs=[]
    with torch.no_grad():
        for b in loader:
            ids=b['input_ids'].to(device); mask=b['attention_mask'].to(device)
            img=b['image'].to(device);     hi=b['has_image'].to(device)
            kg=b['kg_embedding'].to(device);lbl=b['label'].to(device)
            logits=model(ids,mask,img,hi,kg)
            p=torch.softmax(logits,dim=1)[:,1]
            labels.extend(lbl.cpu().tolist())
            preds.extend(logits.argmax(1).cpu().tolist())
            probs.extend(p.cpu().tolist())
    return np.array(labels), np.array(preds), np.array(probs)


def _make_loader(df, batch_size=32):
    return torch.utils.data.DataLoader(
        UnifiedFakeNewsDataset(df,precompute_kg=False),
        batch_size=batch_size, shuffle=False, collate_fn=collate_fn,
        num_workers=2, pin_memory=True
    )


def evaluate_loader(model, loader, device, tag=''):
    y,p,pb=_run_inference(model,loader,device)
    m=compute_metrics(y.tolist(),p.tolist(),pb.tolist())
    if tag: print(f'\n{"="*60}\n  {tag}\n{"="*60}')
    for k,v in m.items(): print(f'  {k:<25} {v:.4f}')
    print(f'  {"n_samples":<25} {len(y)}')
    return m, y, p, pb


def evaluate_by_dataset(model, gc_test_df, wb_test_df, device, batch_size=32):
    """
    Runs evaluation for every group:
    - gossipcop           GossipCop full test (English, text-only)
    - weibo_text_only     Weibo rows with has_image=False
    - weibo_image_only    Weibo rows with has_image=True, input_text zeroed
    - weibo_both          Weibo rows with has_image=True, full text+image
    - weibo_full          All Weibo test rows
    - combined            GossipCop + Weibo combined
    """
    results = {}

    m,y,p,pb = evaluate_loader(model,_make_loader(gc_test_df,batch_size),device,'GossipCop (text-only dataset)')
    results['gossipcop'] = {**m,'n_samples':len(y),'labels':y.tolist(),'preds':p.tolist(),'probs':pb.tolist()}

    wb_no_img = wb_test_df[wb_test_df['has_image']==False].reset_index(drop=True)
    if len(wb_no_img)>0:
        m,y,p,pb = evaluate_loader(model,_make_loader(wb_no_img,batch_size),device,'Weibo - Text-Only (no image)')
        results['weibo_text_only'] = {**m,'n_samples':len(y),'labels':y.tolist(),'preds':p.tolist(),'probs':pb.tolist()}

    wb_img = wb_test_df[wb_test_df['has_image']==True].reset_index(drop=True)
    if len(wb_img)>0:
        wb_txt0 = wb_img.copy(); wb_txt0['input_text']=''
        m,y,p,pb = evaluate_loader(model,_make_loader(wb_txt0,batch_size),device,'Weibo - Image-Only Ablation (text zeroed)')
        results['weibo_image_only'] = {**m,'n_samples':len(y),'labels':y.tolist(),'preds':p.tolist(),'probs':pb.tolist()}

        m,y,p,pb = evaluate_loader(model,_make_loader(wb_img,batch_size),device,'Weibo - Text+Image (both modalities)')
        results['weibo_both'] = {**m,'n_samples':len(y),'labels':y.tolist(),'preds':p.tolist(),'probs':pb.tolist()}

    m,y,p,pb = evaluate_loader(model,_make_loader(wb_test_df,batch_size),device,'Weibo - Full Test Set')
    results['weibo_full'] = {**m,'n_samples':len(y),'labels':y.tolist(),'preds':p.tolist(),'probs':pb.tolist()}

    combined_df=pd.concat([gc_test_df,wb_test_df],ignore_index=True).reset_index(drop=True)
    m,y,p,pb = evaluate_loader(model,_make_loader(combined_df,batch_size),device,'COMBINED (GossipCop + Weibo)')
    results['combined'] = {**m,'n_samples':len(y),'labels':y.tolist(),'preds':p.tolist(),'probs':pb.tolist()}
    return results


def evaluate_adversarial(model, loader, device, eps_fgsm=0.008, eps_pgd=0.016):
    model.eval(); cc=fc=pc=tot=0
    for b in loader:
        ids=b['input_ids'].to(device); mask=b['attention_mask'].to(device)
        img=b['image'].to(device);     hi=b['has_image'].to(device)
        kg=b['kg_embedding'].to(device);lbl=b['label'].to(device)
        with torch.no_grad(): cc+=(model(ids,mask,img,hi,kg).argmax(1)==lbl).sum().item()
        fi=fgsm_attack(img,lbl,model,ids,mask,hi,kg,epsilon=eps_fgsm)
        with torch.no_grad(): fc+=(model(ids,mask,fi,hi,kg).argmax(1)==lbl).sum().item()
        pi=pgd_attack(img,lbl,model,ids,mask,hi,kg,epsilon=eps_pgd,num_iter=10)
        with torch.no_grad(): pc+=(model(ids,mask,pi,hi,kg).argmax(1)==lbl).sum().item()
        tot+=lbl.size(0)
    ca,fa,pa=cc/tot,fc/tot,pc/tot
    return dict(clean_accuracy=ca,fgsm_accuracy=fa,pgd_accuracy=pa,
                fgsm_drop_pct=(ca-fa)/ca*100 if ca>0 else 0,
                pgd_drop_pct=(ca-pa)/ca*100  if ca>0 else 0)


### 11. Results Tabulation and Visualisation

In [11]:
def tabulate_results(all_results, output_dir=OUTPUT_DIR):
    keys=['accuracy','f1_binary','f1_macro','f1_weighted','precision_binary',
          'recall_binary','auc_roc','mcc','cohen_kappa','n_samples']
    header=['Group']+[k.replace('_',' ').title() for k in keys]
    rows=[]
    for g,res in all_results.items():
        row=[g.replace('_',' ').title()]
        for k in keys:
            v=res.get(k,float('nan'))
            row.append(f'{v:.4f}' if isinstance(v,float) else str(v))
        rows.append(row)
    cw=[max(len(h),max(len(r[i]) for r in rows)) for i,h in enumerate(header)]
    sep='+-'+'-+-'.join('-'*w for w in cw)+'-+'
    fmt='| '+' | '.join(f'{{:<{w}}}' for w in cw)+' |'
    lines=[sep,fmt.format(*header),sep]
    for r in rows: lines+=[fmt.format(*r),sep]
    tbl='\n'.join(lines)
    print('\n'+tbl)
    with open(os.path.join(output_dir,'evaluation_table.txt'),'w') as f: f.write(tbl)
    df_out=pd.DataFrame(rows,columns=header)
    df_out.to_csv(os.path.join(output_dir,'evaluation_table.csv'),index=False)
    print(f'\nTable saved to {output_dir}/evaluation_table.txt and .csv')
    return df_out


def plot_training_curve(training_log, output_dir=OUTPUT_DIR):
    ep=[e['epoch'] for e in training_log]
    tl=[e['train_loss'] for e in training_log]
    vl=[e['val_loss']   for e in training_log]
    va=[e['val_acc']    for e in training_log]
    gn=[e.get('mean_grad_norm',0) for e in training_log]
    fig,axs=plt.subplots(1,3,figsize=(18,5))
    axs[0].plot(ep,tl,'o-',label='Train'); axs[0].plot(ep,vl,'s-',label='Val')
    axs[0].set(xlabel='Epoch',ylabel='Loss',title='Loss Curves'); axs[0].legend(); axs[0].grid(True)
    axs[1].plot(ep,va,'^-',color='green'); axs[1].axhline(max(va),color='red',ls='--',label=f'Best={max(va):.4f}')
    axs[1].set(xlabel='Epoch',ylabel='Accuracy',title='Validation Accuracy'); axs[1].legend(); axs[1].grid(True)
    axs[2].plot(ep,gn,'d-',color='orange')
    axs[2].set(xlabel='Epoch',ylabel='Gradient Norm',title='Gradient Norm'); axs[2].grid(True)
    plt.tight_layout()
    p=os.path.join(output_dir,'training_curves.png'); plt.savefig(p,dpi=150,bbox_inches='tight'); plt.close()
    print(f'Training curves: {p}')


def plot_all_roc_curves(all_results, output_dir=OUTPUT_DIR):
    fig,ax=plt.subplots(figsize=(9,7))
    colors=plt.cm.Set1(np.linspace(0,1,len(all_results)))
    for (g,res),col in zip(all_results.items(),colors):
        y=np.array(res.get('labels',[])); pb=np.array(res.get('probs',[]))
        if len(y)<2 or len(set(y))<2: continue
        fpr,tpr,_=roc_curve(y,pb); auc_v=auc(fpr,tpr)
        ax.plot(fpr,tpr,color=col,label=f'{g.replace("_"," ").title()} (AUC={auc_v:.3f})')
    ax.plot([0,1],[0,1],'k--',label='Random')
    ax.set(xlabel='FPR',ylabel='TPR',title='ROC Curves — All Evaluation Groups')
    ax.legend(fontsize=8); ax.grid(True); plt.tight_layout()
    p=os.path.join(output_dir,'roc_curves_all.png'); plt.savefig(p,dpi=150,bbox_inches='tight'); plt.close()
    print(f'ROC curves: {p}')


def plot_all_confusion_matrices(all_results, output_dir=OUTPUT_DIR):
    n=len(all_results); cols=min(3,n); rows_n=int(np.ceil(n/cols))
    fig,axes=plt.subplots(rows_n,cols,figsize=(5*cols,4*rows_n))
    axes=np.array(axes).flatten()
    for i,(g,res) in enumerate(all_results.items()):
        y=np.array(res.get('labels',[])); p=np.array(res.get('preds',[]))
        if len(y)==0: continue
        cm=confusion_matrix(y,p)
        sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=axes[i],
                    xticklabels=['Real','Fake'],yticklabels=['Real','Fake'])
        axes[i].set(title=g.replace('_',' ').title(),xlabel='Predicted',ylabel='True')
    for j in range(i+1,len(axes)): axes[j].set_visible(False)
    plt.suptitle('Confusion Matrices — All Groups',y=1.02)
    plt.tight_layout()
    p=os.path.join(output_dir,'confusion_matrices_all.png'); plt.savefig(p,dpi=150,bbox_inches='tight'); plt.close()
    print(f'Confusion matrices: {p}')


### 12. Standalone Predict Function
Single function handles all input combinations:
- `predict(model, device, text='...', lang='en')` — GossipCop text-only
- `predict(model, device, text='...', lang='zh')` — Weibo text-only
- `predict(model, device, image='path', lang='zh')` — Weibo image-only
- `predict(model, device, text='...', image='path', lang='zh')` — Weibo text+image

In [12]:
def predict(model, device, text=None, image=None, lang='en', kg_linker=None, max_length=128):
    """
    Standalone inference for any input combination.

    Parameters
    ----------
    model      : trained UnifiedMultimodalFakeNewsDetector (or DataParallel wrapper)
    device     : torch.device
    text       : str | None   article or tweet text; None = image-only mode
    image      : str | PIL.Image | None  image file path, PIL image, or None
    lang       : 'en' (GossipCop) | 'zh' (Weibo)
    kg_linker  : KGLinker instance | None  (None = skip KG)
    max_length : tokeniser max length (default 128)

    Returns
    -------
    dict: label, label_id, confidence, prob_real, prob_fake, modality, lang
    """
    model.eval()
    tok = AutoTokenizer.from_pretrained('bert-base-uncased' if lang=='en' else 'bert-base-multilingual-cased')
    if text and str(text).strip():
        enc = tok(str(text),max_length=max_length,padding='max_length',truncation=True,return_tensors='pt')
        ids  = enc['input_ids'].to(device)
        mask = enc['attention_mask'].to(device)
        text_used = True
    else:
        ids  = torch.zeros(1,max_length,dtype=torch.long).to(device)
        mask = torch.zeros(1,max_length,dtype=torch.long).to(device)
        text_used = False

    img_used = False
    img_t  = torch.zeros(1,3,224,224).to(device)
    has_im = torch.tensor([0.],dtype=torch.float32).to(device)
    if image is not None:
        try:
            pil = Image.open(image).convert('RGB') if isinstance(image,str) else image.convert('RGB')
            img_t  = image_transform(pil).unsqueeze(0).to(device)
            has_im = torch.tensor([1.],dtype=torch.float32).to(device)
            img_used = True
        except Exception as e:
            print(f'[predict] Could not load image: {e}')

    kg = kg_linker.get_embedding(str(text)).unsqueeze(0).to(device) if (kg_linker and text and str(text).strip()) else torch.zeros(1,300).to(device)

    with torch.no_grad():
        logits = model(ids,mask,img_t,has_im,kg)
        probs  = torch.softmax(logits,dim=1)[0]

    pr,pf  = probs[0].item(), probs[1].item()
    lid    = int(probs.argmax().item())
    modal  = '+'.join(filter(None,['text' if text_used else None,'image' if img_used else None])) or 'empty'
    return dict(label='fake' if lid==1 else 'real', label_id=lid,
                confidence=round(max(pr,pf),4), prob_real=round(pr,4), prob_fake=round(pf,4),
                modality=modal, lang=lang)


def demo_predict(model, device, gc_test_df=None, wb_test_df=None):
    """Tests predict() with all 4 input combinations and prints results."""
    print('='*65); print('  PREDICT() STANDALONE DEMO'); print('='*65)
    if gc_test_df is not None and len(gc_test_df)>0:
        s=gc_test_df.iloc[0]
        r=predict(model,device,text=s['input_text'],lang='en')
        print(f'\n[1] GossipCop English text-only')
        print(f'    Input   : {str(s["input_text"])[:80]}...')
        print(f'    True    : {"fake" if s["label"]==1 else "real"}')
        print(f'    Predict : {r["label"]} (conf={r["confidence"]:.4f}  modality={r["modality"]})')
    if wb_test_df is not None and len(wb_test_df)>0:
        st=wb_test_df[wb_test_df['has_image']==False].iloc[0] if any(wb_test_df['has_image']==False) else wb_test_df.iloc[0]
        r=predict(model,device,text=st['input_text'],lang='zh')
        print(f'\n[2] Weibo Chinese text-only')
        print(f'    Input   : {str(st["input_text"])[:80]}...')
        print(f'    True    : {"fake" if st["label"]==1 else "real"}')
        print(f'    Predict : {r["label"]} (conf={r["confidence"]:.4f}  modality={r["modality"]})')
        si=wb_test_df[wb_test_df['has_image']==True]
        if len(si)>0:
            si=si.iloc[0]
            r=predict(model,device,image=si['image_path'],lang='zh')
            print(f'\n[3] Weibo image-only')
            print(f'    Image   : {si["image_path"]}')
            print(f'    True    : {"fake" if si["label"]==1 else "real"}')
            print(f'    Predict : {r["label"]} (conf={r["confidence"]:.4f}  modality={r["modality"]})')
            r=predict(model,device,text=si['input_text'],image=si['image_path'],lang='zh')
            print(f'\n[4] Weibo text+image')
            print(f'    Text    : {str(si["input_text"])[:80]}...')
            print(f'    Image   : {si["image_path"]}')
            print(f'    True    : {"fake" if si["label"]==1 else "real"}')
            print(f'    Predict : {r["label"]} (conf={r["confidence"]:.4f}  modality={r["modality"]})')
    print('\n'+'='*65)


### 13. Explainability (Gradient Attribution + LIME)

In [13]:
def run_gradient_attribution(model, test_loader, device, output_dir=OUTPUT_DIR, n_samples=20):
    os.makedirs(output_dir,exist_ok=True); model.eval(); scores=[]
    for batch in test_loader:
        if len(scores)>=n_samples: break
        ids=batch['input_ids'].to(device); mask=batch['attention_mask'].to(device)
        img=batch['image'].to(device).requires_grad_(True)
        hi=batch['has_image'].to(device); kg=batch['kg_embedding'].to(device); lbl=batch['label'].to(device)
        logits=model(ids,mask,img,hi,kg)
        pred=logits.argmax(1)
        for i in range(logits.size(0)):
            if len(scores)>=n_samples: break
            model.zero_grad()
            logits[i,pred[i]].backward(retain_graph=(i<logits.size(0)-1))
            gm=img.grad[i].abs().mean().item() if img.grad is not None else 0.
            scores.append(gm)
            if img.grad is not None: img.grad.zero_()
    fig,ax=plt.subplots(figsize=(10,6))
    ax.barh([f'Sample {i}' for i in range(len(scores))],scores,color='steelblue')
    ax.set(xlabel='Gradient Magnitude (Image Features)',title='Feature Importance via Gradient Attribution')
    plt.tight_layout()
    p=os.path.join(output_dir,'shap_summary.png'); plt.savefig(p,dpi=150,bbox_inches='tight'); plt.close()
    print(f'Gradient attribution: {p}'); return scores


def run_lime(model, test_dataset, device, output_dir=OUTPUT_DIR, n_samples=20):
    os.makedirs(output_dir,exist_ok=True); model.eval()
    tok_en=AutoTokenizer.from_pretrained('bert-base-uncased')
    tok_zh=AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
    explainer=lime.lime_text.LimeTextExplainer(class_names=['real','fake'])
    def pp(texts,lang='en'):
        tok=tok_en if lang=='en' else tok_zh; out=[]
        for t in texts:
            enc=tok(t,max_length=128,padding='max_length',truncation=True,return_tensors='pt')
            ids=enc['input_ids'].to(device); mask=enc['attention_mask'].to(device)
            img=torch.zeros(1,3,224,224).to(device); hi=torch.tensor([0.],dtype=torch.float32).to(device)
            kg=torch.zeros(1,300).to(device)
            with torch.no_grad(): pr=torch.softmax(model(ids,mask,img,hi,kg),dim=1).cpu().numpy()
            out.append(pr[0])
        return np.array(out)
    for idx in range(min(n_samples,len(test_dataset))):
        try:
            row=test_dataset.df.iloc[idx]; lang=row.get('lang','en')
            exp=explainer.explain_instance(row['input_text'],lambda ts:pp(ts,lang),num_features=10,num_samples=500)
            exp.save_to_file(os.path.join(output_dir,f'lime_sample_{idx}.html'))
        except Exception as e:
            logging.getLogger().error(f'LIME failed sample {idx}: {e}')
    print(f'LIME saved to {output_dir}/lime_sample_*.html')


### 14. Main Execution
Run this cell to train, evaluate all groups, generate plots, and run explainability.

In [14]:
def main():
    logger=setup_logging(); set_seed(42)
    logger.info('Starting Multimodal Fake News Detection Pipeline v2')

    print('Preparing datasets...')
    train_df,val_df,test_df,gc_test_df,wb_test_df = build_unified_splits()
    print(f'Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}')
    print(f'  GossipCop test: {len(gc_test_df):,} | Weibo test: {len(wb_test_df):,}')

    set_seed(2024); BATCH_SIZE=32
    print('Building DataLoaders...')
    train_ds=UnifiedFakeNewsDataset(train_df,precompute_kg=False)
    val_ds  =UnifiedFakeNewsDataset(val_df,  precompute_kg=False)
    test_ds =UnifiedFakeNewsDataset(test_df, precompute_kg=False)
    train_loader=torch.utils.data.DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,
                     collate_fn=collate_fn,num_workers=2,pin_memory=True,persistent_workers=True)
    val_loader  =torch.utils.data.DataLoader(val_ds,  batch_size=BATCH_SIZE,shuffle=False,
                     collate_fn=collate_fn,num_workers=2,pin_memory=True,persistent_workers=True)
    test_loader =torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE,shuffle=False,
                     collate_fn=collate_fn,num_workers=2,pin_memory=True,persistent_workers=True)
    print('DataLoaders ready.')

    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Loading model on {device}...')
    model=UnifiedMultimodalFakeNewsDetector().to(device)
    if torch.cuda.device_count()>1:
        print(f'Using {torch.cuda.device_count()} GPUs via DataParallel!'); model=nn.DataParallel(model)
    total_params=sum(p.numel() for p in model.parameters())
    print(f'Model loaded: {total_params:,} parameters')

    print('\nTraining (max 30 epochs, patience 5)...')
    best_val,tlog=train_model(model,train_loader,val_loader,device,max_epochs=30,patience=5)
    print(f'Best val accuracy: {best_val:.4f}')
    plot_training_curve(tlog)

    state=torch.load(os.path.join(OUTPUT_DIR,'best_model.pt'),map_location=device)
    (model.module if hasattr(model,'module') else model).load_state_dict(state)
    print('Best weights loaded.')

    print('\nPer-dataset / per-modality evaluation...')
    all_results=evaluate_by_dataset(model,gc_test_df,wb_test_df,device,batch_size=BATCH_SIZE)
    save_metrics({k:{m:v for m,v in r.items() if m not in ['labels','preds','probs']}
                  for k,r in all_results.items()}, os.path.join(OUTPUT_DIR,'evaluation_results_detailed.json'))

    print('\nAdversarial robustness...')
    adv=evaluate_adversarial(model,test_loader,device)
    for k,v in adv.items(): print(f'  {k:<30} {v:.4f}')
    save_metrics(adv,os.path.join(OUTPUT_DIR,'adversarial_results.json'))

    print('\n'+'='*65+'\n  EVALUATION SUMMARY TABLE\n'+'='*65)
    tabulate_results(all_results)

    print('\nGenerating plots...')
    plot_all_roc_curves(all_results)
    plot_all_confusion_matrices(all_results)

    print('\nExplainability...')
    run_gradient_attribution(model,test_loader,device,n_samples=20)
    run_lime(model,test_ds,device,n_samples=20)
    print('Explainability artifacts saved.')

    demo_predict(model,device,gc_test_df=gc_test_df,wb_test_df=wb_test_df)

    print('\n'+'='*65+'\n  FINAL RESULTS (Combined)\n'+'='*65)
    for k,v in all_results['combined'].items():
        if k not in ['labels','preds','probs']:
            print(f'  {k:<30} {v:.4f}' if isinstance(v,float) else f'  {k:<30} {v}')
    print(f'  {"FGSM acc":<30} {adv["fgsm_accuracy"]:.4f}')
    print(f'  {"PGD acc":<30}  {adv["pgd_accuracy"]:.4f}')
    print(f'  {"Total params":<30} {total_params:,}')
    print('='*65)
    logger.info('Pipeline complete.')

if __name__ == '__main__':
    main()


Preparing datasets...
Train 15,371 | Val 2,420 | Test 2,420
  GossipCop test: 1,226 | Weibo test: 1,194
Building DataLoaders...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 06 | train=0.6347 | val_loss=nan | val_acc=0.5636 | grad=nan


Training:   0%|          | 0/481 [00:00<?, ?it/s]

Epoch 07 | train=0.0000 | val_loss=nan | val_acc=0.5636 | grad=0.000


Training:   0%|          | 0/481 [00:00<?, ?it/s]

Epoch 08 | train=0.0000 | val_loss=nan | val_acc=0.5636 | grad=0.000
Early stopping at epoch 8
Best val accuracy: 0.8434
Training curves: /kaggle/working/outputs/training_curves.png
Best weights loaded.

Per-dataset / per-modality evaluation...

  GossipCop (text-only dataset)
  accuracy                  0.7912
  precision_binary          0.8883
  recall_binary             0.4279
  f1_binary                 0.5776
  precision_macro           0.8305
  recall_macro              0.7005
  f1_macro                  0.7194
  f1_weighted               0.7667
  auc_roc                   0.8391
  mcc                       0.5148
  cohen_kappa               0.4606
  n_samples                 1226

  Weibo - Text-Only (no image)
  accuracy                  0.8873
  precision_binary          0.9333
  recall_binary             0.8235
  f1_binary                 0.8750
  precision_macro           0.8935
  recall_macro              0.8847
  f1_macro                  0.8862
  f1_weighted              

### 15. Resume / Inference from Saved Checkpoint
Uncomment the last line to run evaluation only (skip training).

In [15]:
def load_and_evaluate(model_path=None, batch_size=32):
    """
    Load a saved best_model.pt and run full evaluation without re-training.
    Usage:
        model, results = load_and_evaluate()
    """
    if model_path is None: model_path=os.path.join(OUTPUT_DIR,'best_model.pt')
    assert os.path.isfile(model_path), f'Model not found: {model_path}'
    train_df,val_df,test_df,gc_test_df,wb_test_df=build_unified_splits()
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model=UnifiedMultimodalFakeNewsDetector().to(device)
    if torch.cuda.device_count()>1: model=nn.DataParallel(model)
    state=torch.load(model_path,map_location=device)
    (model.module if hasattr(model,'module') else model).load_state_dict(state)
    print(f'Loaded: {model_path}')
    all_results=evaluate_by_dataset(model,gc_test_df,wb_test_df,device,batch_size=batch_size)
    tabulate_results(all_results)
    plot_all_roc_curves(all_results)
    plot_all_confusion_matrices(all_results)
    demo_predict(model,device,gc_test_df=gc_test_df,wb_test_df=wb_test_df)
    return model, all_results

# Uncomment to run evaluation-only (no training):
model, results = load_and_evaluate()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded: /kaggle/working/outputs/best_model.pt

  GossipCop (text-only dataset)
  accuracy                  0.7912
  precision_binary          0.8883
  recall_binary             0.4279
  f1_binary                 0.5776
  precision_macro           0.8305
  recall_macro              0.7005
  f1_macro                  0.7194
  f1_weighted               0.7667
  auc_roc                   0.8391
  mcc                       0.5148
  cohen_kappa               0.4606
  n_samples                 1226

  Weibo - Text-Only (no image)
  accuracy                  0.8873
  precision_binary          0.9333
  recall_binary             0.8235
  f1_binary                 0.8750
  precision_macro           0.8935
  recall_macro              0.8847
  f1_macro                  0.8862
  f1_weighted               0.8867
  auc_roc                   0.9690
  mcc                       0.7782
  cohen_kappa               0.7732
  n_samples                 71

  Weibo - Image-Only Ablation (text zeroed)
  accuracy